# MVP Analysis

This notebook focuses on the analysis and interepretation of results of performed experiments in the mvp of the diploma thesis.

In [1]:
import os
import sys

sys.dont_write_bytecode = True

In [8]:
import json
import copy
import random
import datetime
import numpy as np
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
import pprint
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from dataclasses import asdict
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
project_pwd = Path().cwd().parents[1]
project_pwd = os.path.abspath(project_pwd)
if project_pwd not in sys.path:
    sys.path.append(project_pwd)

sys.dont_write_bytecode = True

In [4]:
from scripts.intro.model import *
from scripts.intro.loader import *
from scripts.intro.block import *
from scripts.intro.activation import *
from scripts.intro.replacement import *
from scripts.intro.eval import *
from scripts.intro.metrics import *

from scripts.utils import *
from configs.intro_config import *

In [5]:
mcfg = MCFG()
torch.manual_seed(mcfg.seed)

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

In [7]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(DEVICE)

2.11.0+cu128
12.8
True
cuda


### Analyses

load logs

In [16]:
def load_json(filepath):
    with open(filepath) as f:
        return json.load(f)

In [17]:
log = "mvp_log_1.json"
dirpath = "../../data/mvp/results/logs/"

filepath = os.path.join(dirpath, log)

In [18]:
data = load_json(filepath)

In [22]:
print(data['metadata'])

{'time': '2026-05-06'}


In [23]:
data

{'metadata': {'time': '2026-05-06'},
 'base_loss': 3.0152251720428467,
 'base_ppl': 20.393682510735303,
 '1': {'name': 'random_k_result_one_shot_no_recovery',
  'cfg': {'seq_len': 128,
   'batch_size': 2,
   'k_blocks': 5,
   'selection_strategy': 'random_k',
   'bi_rank_order': 'desc',
   'replacement_strategy': 'one_shot',
   'replacement_operator': 'linear',
   'calib_split': 'train[:5%]',
   'eval_split': 'validation[:1%]',
   'num_calib_batches': 24,
   'num_eval_batches': 24,
   'replacement_epochs': 5,
   'replacement_lr': 0.001,
   'replacement_batch_size': 2048,
   'seed': 21},
  'result': {'strategy': 'one_shot',
   'model_loss': 3.8500702381134033,
   'model_ppl': 46.99636405160404,
   'logs': [{'layer_idx': 5,
     'layer_path': 'model.layers.5.mlp',
     'new_operator_mse': 0.5351581573486328},
    {'layer_idx': 9,
     'layer_path': 'model.layers.9.mlp',
     'new_operator_mse': 1.5139983892440796},
    {'layer_idx': 13,
     'layer_path': 'model.layers.13.mlp',
     'new

In [24]:
data

{'metadata': {'time': '2026-05-06'},
 'base_loss': 3.0152251720428467,
 'base_ppl': 20.393682510735303,
 '1': {'name': 'random_k_result_one_shot_no_recovery',
  'cfg': {'seq_len': 128,
   'batch_size': 2,
   'k_blocks': 5,
   'selection_strategy': 'random_k',
   'bi_rank_order': 'desc',
   'replacement_strategy': 'one_shot',
   'replacement_operator': 'linear',
   'calib_split': 'train[:5%]',
   'eval_split': 'validation[:1%]',
   'num_calib_batches': 24,
   'num_eval_batches': 24,
   'replacement_epochs': 5,
   'replacement_lr': 0.001,
   'replacement_batch_size': 2048,
   'seed': 21},
  'result': {'strategy': 'one_shot',
   'model_loss': 3.8500702381134033,
   'model_ppl': 46.99636405160404,
   'logs': [{'layer_idx': 5,
     'layer_path': 'model.layers.5.mlp',
     'new_operator_mse': 0.5351581573486328},
    {'layer_idx': 9,
     'layer_path': 'model.layers.9.mlp',
     'new_operator_mse': 1.5139983892440796},
    {'layer_idx': 13,
     'layer_path': 'model.layers.13.mlp',
     'new